# Day 7 | Hands-On 2: Build Gold Layer — `fact_sales`
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Grain** | One row per order line item (`order_items`) |
| **Source** | `silver.order_items`, `silver.orders`, `silver.products`, `silver.payments`, `silver.address`, `gold.dim_date` |
| **Target** | `gbmart.gold.fact_sales` |
| **Duration** | ~2.5 hours (2 sessions) |

### Shape — the real, running GlobalMart fact table
```
fact_sales
├── fact_sales_sk        (surrogate PK -- the ONLY generated key in this table)
├── Payment_ID           (natural key, from silver.payments)
├── Customer_ID          (natural key)
├── Product_ID           (natural key)
├── Order_ID             (natural key)
├── Address_ID           (natural key)
├── Time_ID              (natural key -- dim_date.date_key)
├── Quantity_purchased   (measure)
├── Actual_price         (measure)
├── Discounted_price     (measure)
└── Sales_amount         (measure = Quantity_purchased × Discounted_price)
```

> **Reminder from ILT 1:** every FK here is a natural/business key, not a dimension surrogate key — a deliberate, documented simplification for this training build. `dim_customer`/`dim_product` still have their own proper SCD2 surrogate keys; this fact table just isn't wired to them that way.

### Learning Objectives
- Build a real, multi-table Gold fact table step by step, verifying row counts after every join
- Handle the address one-to-many trap correctly (from ILT 1) inside a real build
- Make a fact-table cell safely re-runnable without restarting from scratch
- Serve the finished fact table through real, business-facing Gold views

---
**Instructions:** Run each cell with **Shift + Enter**, in order. Watch the printed row counts after every join — if a count jumps unexpectedly, stop and re-check that join before continuing, don't push through to the end and debug backwards.

---
## Step 1 — Setup

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
print(f"Schema '{CATALOG}.gold' is ready -- your 6 dimensions from Day 6 already live here.")

---
## Step 2 — Read the Grain + Order Header Info
`order_items` is the grain. Join `orders` to get `Customer_ID` and `order_date` (needed later to look up `Time_ID`).

In [ ]:
# order_item_id is the grain's own identity -- it's what fact_sales_sk
# gets hashed from at the very end, so we keep it through every step.
order_items_df = spark.table("gbmart.silver.order_items") \
    .select(
        col("order_item_id"),
        col("order_id").alias("Order_ID"),
        col("product_id").alias("Product_ID"),
        col("quantity").alias("Quantity_purchased")
    )

# Pulling customer_id and order_date from orders now, since order_items
# itself has neither -- both are needed for later steps (address lookup
# needs Customer_ID, dim_date lookup needs order_date).
orders_df = spark.table("gbmart.silver.orders") \
    .select(col("order_id").alias("Order_ID"), col("customer_id").alias("Customer_ID"), "order_date")

base_df = order_items_df.join(orders_df, "Order_ID")
print(f"Base rows: {base_df.count():,}")
print("This number should match silver.order_items exactly -- every order_item has exactly one parent order.")

---
## Step 3 — Look Up `Actual_price` and `Discounted_price`
Join `silver.products` on the natural `Product_ID`, filtered to `is_current = true` so each product resolves to exactly one price row (per ILT 1's warning about SCD2 dimensions fanning out a join).

In [ ]:
products_current = spark.table("gbmart.silver.products").filter("is_current = true") \
    .select(
        col("product_id").alias("Product_ID"),
        col("actual_price_inr").alias("Actual_price"),
        col("discounted_price_inr").alias("Discounted_price")
    )

enriched_df = base_df.join(products_current, "Product_ID")
print(f"Rows after products join: {enriched_df.count():,}")
print("If this is LOWER than Step 2's count, some order_items reference a product_id that")
print("doesn't exist (or isn't current) in silver.products -- worth investigating, not ignoring.")

---
## Step 4 — Look Up `Time_ID`
Join on `order_date` to resolve `dim_date.date_key`, used here as `Time_ID`. Recall from Day 6: `dim_date`'s range was generated to exactly cover GlobalMart's real order dates, so this join should not lose rows.

In [ ]:
dim_date_df = spark.table("gbmart.gold.dim_date").select("date", col("date_key").alias("Time_ID"))

# Drop any leftover Time_ID/date columns from a previous run of this cell --
# makes this cell safe to re-run (e.g. after fixing something upstream)
# without needing to restart the whole notebook from Step 2.
enriched_df = enriched_df.drop("Time_ID", "date")

enriched_df = enriched_df.join(dim_date_df, enriched_df.order_date == dim_date_df.date, "left")
print(f"Rows after dim_date join: {enriched_df.count():,}")
print(f"Rows with no matching Time_ID: {enriched_df.filter(col('Time_ID').isNull()).count():,}")
print("This should be 0 -- dim_date's range was built from these exact order dates in Day 6.")

---
## Step 5 — Look Up `Address_ID` (the One-to-Many Trap from ILT 1)
`silver.address` links to `Customer_ID`, not to a specific order — a customer can have more than one address (Billing, Shipping, or both). Exactly like ILT 1 demonstrated: rank per customer, prefer `Shipping`, take the first one available.

In [ ]:
address_window = Window.partitionBy("customer_id").orderBy(
    when(col("address_type").contains("Shipping"), 0).otherwise(1)
)

address_primary = spark.table("gbmart.silver.address") \
    .withColumn("_rank", row_number().over(address_window)) \
    .filter("_rank = 1") \
    .select(col("customer_id").alias("Customer_ID"), col("address_id").alias("Address_ID"))

enriched_df = enriched_df.join(address_primary, "Customer_ID", "left")
print(f"Rows after address join: {enriched_df.count():,}")
print(f"Rows with no matching Address_ID: {enriched_df.filter(col('Address_ID').isNull()).count():,}")
print("Compare this row count to Step 4's -- it must be UNCHANGED. If it grew, the address")
print("ranking above isn't collapsing multi-address customers down to one row each -- go back and check it.")

---
## Step 6 — Look Up `Payment_ID`
Payments are 1:1 with orders — join `silver.payments` on `Order_ID` to get the natural `Payment_ID` directly. No ranking needed here, unlike Step 5, because this relationship really is one-to-one.

In [ ]:
payments_df = spark.table("gbmart.silver.payments") \
    .select(col("order_id").alias("Order_ID"), col("payment_id").alias("Payment_ID"))

enriched_df = enriched_df.join(payments_df, "Order_ID", "left")
print(f"Rows after payments join: {enriched_df.count():,}")
print(f"Rows with no matching Payment_ID: {enriched_df.filter(col('Payment_ID').isNull()).count():,}")
print("Again, this row count must match Step 5's exactly -- a 1:1 join should never change row count.")

---
## Step 7 — Compute `Sales_amount` and Final Select
`fact_sales_sk` is the one and only surrogate key in this table — generated from `order_item_id`, since that's the fact's natural grain key.

In [ ]:
fact_sales_df = enriched_df \
    .withColumn("fact_sales_sk", sha2(col("order_item_id"), 256)) \
    .withColumn("Sales_amount", col("Quantity_purchased") * col("Discounted_price")) \
    .select(
        "fact_sales_sk", "Payment_ID", "Customer_ID", "Product_ID", "Order_ID", "Address_ID",
        "Time_ID", "Quantity_purchased", "Actual_price", "Discounted_price", "Sales_amount"
    )

print(f"fact_sales rows: {fact_sales_df.count():,}")
fact_sales_df.display()

---
## Step 8 — Write to Gold

In [ ]:
# overwrite mode -- per ILT 1, this is the simple, always-correct strategy
# at GlobalMart's current volume. Incremental MERGE-based refresh replaces
# this in Day 9-10, once you've felt why a full rebuild gets wasteful.
fact_sales_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.fact_sales")
print(f"Written: {spark.table('gbmart.gold.fact_sales').count():,} rows")

---
## Step 9 — Serve It: Business-Facing Gold Views
`fact_sales` on its own isn't what downstream analysts query directly. Real recurring business questions get answered by **views** built on top of it — one agreed-upon definition everyone reports against, instead of every analyst writing a slightly different join and getting slightly different numbers.

### Why a VIEW, not a materialized/aggregate table?
At `fact_sales`'s current size, a `VIEW` recomputes fresh on every query — no separate refresh job to schedule or monitor, and it always reflects whatever's currently in `fact_sales`. If a dashboard ever gets slow at much larger scale, *that's* the point where you'd promote one to a materialized table — not before.

In [ ]:
# View 1 -- who runs this daily: category/merchandising managers.
# Question it answers: "How is each product category trending, month over
# month -- and which categories are leaning on discounts to move volume?"
spark.sql("""
CREATE OR REPLACE VIEW gbmart.gold.vw_monthly_category_sales AS
SELECT
    d.year,
    d.month,
    p.category,
    p.sub_category,
    SUM(f.Quantity_purchased)                          AS total_quantity_sold,
    SUM(f.Sales_amount)                                AS total_revenue,
    COUNT(DISTINCT f.Order_ID)                         AS total_orders,
    ROUND(AVG(f.Actual_price - f.Discounted_price), 2) AS avg_discount_given
FROM gbmart.gold.fact_sales f
JOIN gbmart.gold.dim_product p ON f.Product_ID = p.product_id AND p.is_current = true
JOIN gbmart.gold.dim_date d    ON f.Time_ID = d.date_key
GROUP BY d.year, d.month, p.category, p.sub_category
""")
print("vw_monthly_category_sales created")

spark.sql("""
    SELECT category, SUM(total_revenue) AS yearly_revenue, SUM(total_orders) AS yearly_orders
    FROM gbmart.gold.vw_monthly_category_sales
    GROUP BY category
    ORDER BY yearly_revenue DESC
""").display()

In [ ]:
# View 2 -- who runs this daily: regional/logistics ops leads.
# Question it answers: "Which states/cities drive the most revenue and
# orders, and how big is the average order in each region?" A region
# with high order count but low average order value tells a very
# different story than one with few orders but a high average.
spark.sql("""
CREATE OR REPLACE VIEW gbmart.gold.vw_regional_sales AS
SELECT
    a.state,
    a.city,
    COUNT(DISTINCT f.Order_ID)                                 AS total_orders,
    COUNT(DISTINCT f.Customer_ID)                              AS total_customers,
    SUM(f.Quantity_purchased)                                  AS total_quantity_sold,
    SUM(f.Sales_amount)                                        AS total_revenue,
    ROUND(SUM(f.Sales_amount) / COUNT(DISTINCT f.Order_ID), 2) AS avg_order_value
FROM gbmart.gold.fact_sales f
JOIN gbmart.gold.dim_address a ON f.Address_ID = a.address_id
GROUP BY a.state, a.city
""")
print("vw_regional_sales created")

spark.sql("""
    SELECT state, SUM(total_revenue) AS state_revenue, SUM(total_orders) AS state_orders,
           ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS state_avg_order_value
    FROM gbmart.gold.vw_regional_sales
    GROUP BY state
    ORDER BY state_revenue DESC
    LIMIT 10
""").display()

---
## Key Takeaways

1. **Verify the row count after every single join** — a fact-table build with 6+ joins is exactly where silent fan-outs hide, and they're much cheaper to catch mid-build than after the table is already "done."
2. **`is_current = true` filters are not optional** wherever you join to an SCD2 dimension from a fact table.
3. **Natural-key joins were a deliberate choice, not a shortcut** — you now know exactly what it costs (no point-in-time accuracy) and why it was still the right call for this program.
4. **These two views ARE the actual deliverable** for most downstream consumers — nobody outside the data team queries `fact_sales` directly day to day.

### Submission Checklist
- [ ] Every step's printed row count reviewed — no unexplained jumps
- [ ] `gbmart.gold.fact_sales` written successfully
- [ ] Both `vw_monthly_category_sales` and `vw_regional_sales` created and queried
- [ ] Screenshot of the Step 8 write confirmation + one view's output for submission

---
## Reset (if needed)

In [ ]:
# spark.sql("DROP VIEW IF EXISTS gbmart.gold.vw_monthly_category_sales")
# spark.sql("DROP VIEW IF EXISTS gbmart.gold.vw_regional_sales")
# spark.sql("DROP TABLE IF EXISTS gbmart.gold.fact_sales")
# print("Reset complete")